# Week 1, Day 1: Data Acquisition & Exploration
**EMA Crossover ML Project**

## Goals for Today (2 hours):
- ✅ Fetch labeled signals from Supabase (500+ records)
- ✅ Load data into pandas, check shape and dtypes
- ✅ Initial exploration: `.info()`, `.describe()`, `.head()`
- ✅ Identify missing values, outliers, data types
- ✅ **Deliverable:** Jupyter notebook with data loaded and explored

## 1. Setup & Imports

In [ ]:
# Install required packages (run once)
%pip install pandas numpy matplotlib seaborn requests python-dotenv


In [ ]:
import os
from datetime import datetime

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import seaborn as sns
from dotenv import load_dotenv

import warnings
warnings.filterwarnings('ignore')
load_dotenv()

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)
pd.set_option('display.width', None)

print('Imports successful!')
print(f'Pandas version: {pd.__version__}')
print(f'NumPy version: {np.__version__}')


## 2. Connect to Supabase & Fetch Data

In [ ]:
# Load Supabase credentials from .env
SUPABASE_URL = os.getenv('SUPABASE_URL')
SUPABASE_KEY = os.getenv('SUPABASE_KEY')

if not SUPABASE_URL or not SUPABASE_KEY:
    raise ValueError('Missing SUPABASE_URL or SUPABASE_KEY in your .env file')

headers = {
    'apikey': SUPABASE_KEY,
    'Authorization': f'Bearer {SUPABASE_KEY}',
}

print('Supabase credentials loaded from .env')


In [ ]:
# Fetch all analyzed signals from Supabase REST API using pagination
print('Fetching data from Supabase...')

try:
    batch_size = 1000
    offset = 0
    all_rows = []

    while True:
        response = requests.get(
            f'{SUPABASE_URL}/rest/v1/signals',
            headers={**headers, 'Range-Unit': 'items', 'Range': f'{offset}-{offset + batch_size - 1}'},
            params={'select': '*', 'status': 'eq.analyzed', 'order': 'checked_at_utc.asc'},
            timeout=30,
        )
        response.raise_for_status()

        batch = response.json()
        if not batch:
            break

        all_rows.extend(batch)
        print(f'Fetched {len(all_rows):,} rows so far...')

        if len(batch) < batch_size:
            break

        offset += batch_size

    df = pd.DataFrame(all_rows)

    print('Data fetched successfully!')
    print(f'Total records: {len(df):,}')

except Exception as e:
    print(f'Error fetching data: {e}')
    df = None


## 3. Initial Data Inspection

In [ ]:
# Check if data loaded successfully
if df is not None and len(df) > 0:
    print('DATA LOADED SUCCESSFULLY!\n')
    print(f'Shape: {df.shape}')
    print(f'Rows: {df.shape[0]} | Columns: {df.shape[1]}')
else:
    print('No data loaded. Check your .env file and Supabase access.')


In [ ]:
# Display first few rows
print("First 5 rows of data:\n")
df.head()

In [ ]:
# Display last few rows
print("Last 5 rows of data:\n")
df.tail()

## 4. Data Types & Structure (`.info()`)

In [ ]:
# Get detailed info about the DataFrame
print("DataFrame Information:\n")
df.info()

In [ ]:
# Count data types
print("\nColumn types breakdown:")
print(df.dtypes.value_counts())
print("\n" + "="*50)

# Separate columns by type
numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
object_cols = df.select_dtypes(include=['object']).columns.tolist()
datetime_cols = df.select_dtypes(include=['datetime64']).columns.tolist()
bool_cols = df.select_dtypes(include=['bool']).columns.tolist()

print(f"\n📊 Numeric columns: {len(numeric_cols)}")
print(f"📝 Object (string) columns: {len(object_cols)}")
print(f"📅 Datetime columns: {len(datetime_cols)}")
print(f"✓ Boolean columns: {len(bool_cols)}")

## 5. Fix Data Types

In [ ]:
# Convert datetime columns to proper datetime format
datetime_columns = ['checked_at_utc', 'time_of_max_price', 'time_of_min_price']

for col in datetime_columns:
    if col in df.columns:
        df[col] = pd.to_datetime(df[col], utc=True)
        print(f"✅ Converted {col} to datetime")

# Sort by time (oldest to newest) - CRITICAL for time-series ML
df = df.sort_values('checked_at_utc').reset_index(drop=True)
print("\n✅ Data sorted by time (oldest → newest)")

print(f"\nDate range: {df['checked_at_utc'].min()} to {df['checked_at_utc'].max()}")

## 6. Statistical Summary (`.describe()`)

In [ ]:
# Get statistical summary of numeric columns
print("Statistical Summary of All Numeric Features:\n")
df.describe().T  # Transpose for better readability

In [ ]:
# Categorical/String columns summary
print("\nCategorical Columns Summary:\n")
for col in object_cols:
    print(f"\n{col}:")
    print(df[col].value_counts())
    print("-" * 40)

## 7. Missing Values Analysis

In [ ]:
# Count missing values
missing = df.isnull().sum()
missing_pct = (df.isnull().sum() / len(df)) * 100

missing_df = pd.DataFrame({
    'Missing_Count': missing,
    'Missing_Percentage': missing_pct
}).sort_values('Missing_Count', ascending=False)

# Show only columns with missing values
missing_df = missing_df[missing_df['Missing_Count'] > 0]

if len(missing_df) > 0:
    print("⚠️ Columns with Missing Values:\n")
    print(missing_df)
    print(f"\nTotal columns with missing data: {len(missing_df)}")
else:
    print("✅ No missing values found! Clean dataset.")

In [ ]:
# Visualize missing values
if len(missing_df) > 0:
    plt.figure(figsize=(12, 6))
    missing_df['Missing_Percentage'].plot(kind='barh', color='crimson')
    plt.xlabel('Missing Percentage (%)')
    plt.title('Missing Values by Column')
    plt.tight_layout()
    plt.show()
else:
    print("No missing values to visualize.")

## 8. Outlier Detection (Initial)

In [ ]:
# Define the 35 ML features (excluding metadata and labels)
feature_cols = [
    # Signal Context
    'signal_gap_hours',
    # EMA Indicators (7)
    'ema_fast_ltf', 'ema_slow_ltf', 'ema_fast_slope', 'ema_slow_slope',
    'ema_separation', 'price_above_both_emas', 'crossover_candle_strength',
    # Trend Strength (5)
    'adx_ltf', 'adx_slope', 'adx_4h', 'macd_histogram_ltf', 'macd_histogram_4h',
    # Higher Timeframe Alignment (4)
    'htf_4h_bias', 'htf_1d_bias', 'ema_separation_4h', 'rsi_4h',
    # Momentum (2)
    'rsi_ltf', 'roc_ltf',
    # Volatility (4)
    'atr_ltf', 'atr_pct', 'bb_width_ltf', 'price_to_atr',
    # Volume (3)
    'volume_ratio', 'volume_trend', 'crossover_volume_ratio',
    # Market Context (4)
    'fear_greed_index', 'btc_trend_bias', 'hour_of_day', 'day_of_week',
    # Trade Management (3)
    'swing_high', 'swing_low', 'atr_stop_distance'
]

# Filter to only existing columns
existing_features = [col for col in feature_cols if col in df.columns]
print(f"Found {len(existing_features)} out of 35 feature columns\n")

if len(existing_features) < len(feature_cols):
    missing_features = set(feature_cols) - set(existing_features)
    print(f"⚠️ Missing features: {missing_features}")

In [ ]:
# Check for extreme outliers using IQR method
print("Outlier Detection (IQR Method):\n")

numeric_features = [col for col in existing_features if col in numeric_cols]
outlier_summary = []

for col in numeric_features:
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 3 * IQR  # 3*IQR for extreme outliers
    upper_bound = Q3 + 3 * IQR
    
    outliers = ((df[col] < lower_bound) | (df[col] > upper_bound)).sum()
    outlier_pct = (outliers / len(df)) * 100
    
    if outliers > 0:
        outlier_summary.append({
            'Feature': col,
            'Outliers': outliers,
            'Percentage': f"{outlier_pct:.2f}%",
            'Min': df[col].min(),
            'Max': df[col].max()
        })

if outlier_summary:
    outlier_df = pd.DataFrame(outlier_summary).sort_values('Outliers', ascending=False)
    print(outlier_df.to_string(index=False))
else:
    print("✅ No extreme outliers detected!")

## 9. Label Distribution Analysis

In [ ]:
# Analyze the label columns (outcome data)
label_cols = ['max_price_after', 'min_price_after', 'max_move_up_pct', 
              'max_move_down_pct', 'candles_to_max_price', 'candles_to_min_price']

existing_labels = [col for col in label_cols if col in df.columns]

print("Label Columns Summary:\n")
print(df[existing_labels].describe().T)

In [ ]:
# Visualize label distributions
fig, axes = plt.subplots(2, 3, figsize=(15, 10))
axes = axes.flatten()

for idx, col in enumerate(existing_labels):
    if idx < len(axes):
        axes[idx].hist(df[col].dropna(), bins=50, color='steelblue', edgecolor='black')
        axes[idx].set_title(col)
        axes[idx].set_xlabel('Value')
        axes[idx].set_ylabel('Frequency')
        axes[idx].grid(True, alpha=0.3)

plt.tight_layout()
plt.suptitle('Distribution of Label Columns', y=1.02, fontsize=14, fontweight='bold')
plt.show()

## 10. Signal Distribution

In [ ]:
# Count signals by type
print("Signal Type Distribution:\n")
signal_counts = df['signal'].value_counts()
print(signal_counts)
print(f"\nLONG signals: {signal_counts.get('LONG', 0)} ({signal_counts.get('LONG', 0)/len(df)*100:.1f}%)")
print(f"SHORT signals: {signal_counts.get('SHORT', 0)} ({signal_counts.get('SHORT', 0)/len(df)*100:.1f}%)")

In [ ]:
# Signals by symbol
print("\nSignals by Trading Pair:\n")
symbol_counts = df['symbol'].value_counts()
print(symbol_counts)

# Visualize
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Signal type
signal_counts.plot(kind='bar', ax=axes[0], color=['green', 'red'])
axes[0].set_title('Signal Type Distribution')
axes[0].set_xlabel('Signal Type')
axes[0].set_ylabel('Count')
axes[0].grid(True, alpha=0.3)

# Symbol distribution
symbol_counts.plot(kind='barh', ax=axes[1], color='steelblue')
axes[1].set_title('Signals by Trading Pair')
axes[1].set_xlabel('Count')
axes[1].set_ylabel('Symbol')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 11. Summary Report

In [ ]:
print("="*60)
print("📊 DATA EXPLORATION SUMMARY REPORT")
print("="*60)
print(f"\n✅ Total Records: {len(df):,}")
print(f"✅ Total Features: {df.shape[1]}")
print(f"✅ Numeric Features: {len(numeric_cols)}")
print(f"✅ Boolean Features: {len(bool_cols)}")
print(f"✅ Categorical Features: {len(object_cols)}")
print(f"✅ Datetime Features: {len(datetime_cols)}")

print(f"\n📅 Date Range:")
print(f"   From: {df['checked_at_utc'].min()}")
print(f"   To:   {df['checked_at_utc'].max()}")
print(f"   Span: {(df['checked_at_utc'].max() - df['checked_at_utc'].min()).days} days")

print(f"\n🎯 Signals:")
print(f"   LONG signals:  {signal_counts.get('LONG', 0):,}")
print(f"   SHORT signals: {signal_counts.get('SHORT', 0):,}")

print(f"\n💹 Trading Pairs: {df['symbol'].nunique()}")
print(f"   {', '.join(df['symbol'].unique())}")

if len(missing_df) > 0:
    print(f"\n⚠️ Missing Values: {len(missing_df)} columns have missing data")
else:
    print(f"\n✅ Missing Values: None! Clean dataset.")

if outlier_summary:
    print(f"\n⚠️ Outliers Detected: {len(outlier_summary)} features have extreme values")
else:
    print(f"\n✅ Outliers: No extreme outliers detected")

print("\n" + "="*60)
print("🎉 DAY 1 COMPLETE - Data Successfully Loaded & Explored!")
print("="*60)
print("\nNext Steps (Day 2):")
print("  1. Handle missing values (if any)")
print("  2. Remove duplicates (if any)")
print("  3. Feature validation & corrections")
print("  4. Save cleaned dataset for modeling")

## 12. Save Checkpoint (Optional)

In [ ]:
# Save the raw data for future use
# df.to_csv('data_raw.csv', index=False)
# print("✅ Raw data saved to 'data_raw.csv'")

# Or save as pickle (preserves data types)
# df.to_pickle('data_raw.pkl')
# print("✅ Raw data saved to 'data_raw.pkl'")

---
## ✅ Day 1 Checklist:
- [x] Fetch labeled signals from Supabase (500+ records)
- [x] Load data into pandas, check shape and dtypes
- [x] Initial exploration: `.info()`, `.describe()`, `.head()`
- [x] Identify missing values, outliers, data types
- [x] **Deliverable:** Jupyter notebook with data loaded and explored

**Next Session:** Week 1, Day 2 - Data Cleaning